# Source URL Investigation

**Primary author:** Victoria Winters

**Builds on:**
- *DATA_RAW.md* (Victoria — source-by-source `source_url` / `puzzle_name` format documentation, §4.3)
- *notebooks/puzzle_metadata.ipynb* (Victoria — column loading conventions and per-source extraction rules)

**Prompt engineering:** Victoria
**AI assistance:** Claude / Claude Code (Anthropic)
**Environment:** Local

Investigates whether `source_url` from `data/clues_raw.csv` can serve as a reliable grouping key identifying which puzzle each clue belongs to — ideally, one unique `source_url` value per puzzle, consistent across every row from that puzzle. Checks per-source null rates, intra-source formatting drift (scheme, trailing slash, casing, prefix anomalies), one-to-one correspondence with `puzzle_name` and `puzzle_date`, cross-source collisions, and the source-specific edge cases flagged in `DATA_RAW.md §4.3`. Reads `clues_raw.csv` only; writes no files. This is an investigation notebook — its findings feed a later `assign_ids.py` design decision, but no assignment logic is committed here.

---

## §0 — Setup

Loads the five columns required for the investigation: `clue_id`, `source`, `puzzle_name`, `source_url`, and `puzzle_date`. `keep_default_na=False` with `na_values=[""]` is the project-wide convention so the crossword entry "nan" is never silently coerced to `NaN` — not strictly relevant for the columns here, but kept for consistency with the rest of the pipeline. No index is set; natural row order is preserved for downstream `groupby` work.

In [ ]:
# === Imports, Paths, and Load ===
from pathlib import Path

import pandas as pd

DATA_DIR = Path("..") / "data"
INPUT_PATH = DATA_DIR / "clues_raw.csv"

df = pd.read_csv(
    INPUT_PATH,
    usecols=["clue_id", "source", "puzzle_name", "source_url", "puzzle_date"],
    keep_default_na=False,
    na_values=[""],
)
print(f"Loaded {len(df):,} rows from {INPUT_PATH}")
print("\nRows per source:")
print(df["source"].value_counts())

---

## §1 — Null Rates of `source_url` Per Source

For each source, count rows whose `source_url` is null and report both the absolute count and the percentage. Any source whose null rate exceeds 1% is flagged — at that level the column is unreliable as a grouping key without a documented fallback. A very small stray-null rate is tolerable and would be handled row by row.

In [ ]:
# === Null Rates of source_url Per Source ===
null_stats = (
    df.assign(_is_null=df["source_url"].isna())
    .groupby("source")
    .agg(total=("_is_null", "size"), nulls=("_is_null", "sum"))
)
null_stats["pct_null"] = 100 * null_stats["nulls"] / null_stats["total"]
null_stats = null_stats.sort_values("pct_null", ascending=False)
print(null_stats)

flagged = null_stats[null_stats["pct_null"] > 1]
if len(flagged) > 0:
    print(f"\nFlagged sources (>1% null): {list(flagged.index)}")
else:
    print("\nNo sources exceed the 1% null threshold.")

---

## §2 — Formatting Consistency Within Source

Before treating `source_url` as a grouping key, look for cosmetic variations that would cause the *same* logical puzzle to appear as two distinct group keys. Three known patterns to check:

1. **Scheme drift** — some rows `http://`, others `https://` for the same host.
2. **Trailing slash** — some URLs end with `/` and some do not.
3. **Capitalization** — any uppercase in what would otherwise be a lowercase host/path.

On top of those, the value counts of URL prefixes (first ~40 characters) are shown per source to surface any other anomalies visible at a glance. These are diagnostics, not filters.

In [ ]:
# === Scheme, Trailing Slash, and Casing by Source ===

def scheme(u):
    if pd.isna(u):
        return "<null>"
    if u.startswith("https://"):
        return "https"
    if u.startswith("http://"):
        return "http"
    return "other"

fmt = df.assign(
    _scheme=df["source_url"].apply(scheme),
    _trailing_slash=df["source_url"].str.endswith("/").fillna(False),
    _has_upper=df["source_url"].str.contains(r"[A-Z]", regex=True, na=False),
)

print("Scheme counts by source:")
print(fmt.groupby(["source", "_scheme"]).size().unstack(fill_value=0))

print("\nTrailing-slash counts by source:")
print(fmt.groupby(["source", "_trailing_slash"]).size().unstack(fill_value=0))

print("\nContains-uppercase counts by source (True = URL has >=1 uppercase letter):")
print(fmt.groupby(["source", "_has_upper"]).size().unstack(fill_value=0))

In [ ]:
# === Top source_url Prefixes Per Source ===
# Shows the most common first-40-character prefix per source to surface
# unexpected hosts, schemes, or path patterns not caught by the checks above.
for src, sub in df.groupby("source"):
    vc = (
        sub["source_url"]
        .fillna("<NULL>")
        .str.slice(0, 40)
        .value_counts()
        .head(5)
    )
    print(f"\n--- {src} ({len(sub):,} rows) ---")
    print(vc)

---

## §3 — One-to-One Check: `source_url` ↔ puzzle

For `source_url` to work as a grouping key, every row of a given puzzle must share a single `source_url` value, and conversely that `source_url` must not be shared between puzzles. Three passes:

1. **Cardinality** — unique `source_url` count per source and the average `rows_per_url` (should roughly match the typical clues-per-puzzle figure, ~28).
2. **`source_url` → `puzzle_name`** — if any URL maps to more than one distinct `puzzle_name`, the URL is not puzzle-unique (two puzzles share a group key).
3. **`puzzle_name` → `source_url`** — if any `puzzle_name` maps to more than one distinct URL, a single puzzle would be split into multiple groups (typically a symptom of the formatting drift checked in §2).

A fourth pass checks `source_url` → `puzzle_date`. `puzzle_date` NaNs are dropped before counting (common in `thebrowser`), so a URL mapping to one real date plus NaN is treated as consistent, not as a conflict.

In [ ]:
# === Unique source_url Count Per Source ===
non_null = df.dropna(subset=["source_url"]).copy()
uniq = non_null.groupby("source").agg(
    rows=("source_url", "size"),
    unique_urls=("source_url", "nunique"),
)
uniq["rows_per_url"] = uniq["rows"] / uniq["unique_urls"]
print(uniq.sort_values("rows_per_url", ascending=False))

In [ ]:
# === source_url -> puzzle_name Multiplicity ===
url_to_name = (
    non_null.groupby(["source", "source_url"])["puzzle_name"]
    .nunique(dropna=True)
    .reset_index(name="n_names")
)
bad = url_to_name[url_to_name["n_names"] > 1]
print(f"source_url values mapping to >1 distinct puzzle_name: {len(bad):,}")
if len(bad) > 0:
    print("\nBreakdown by source:")
    print(bad.groupby("source").size())

    print("\nExample collisions (first 10):")
    ex = (
        bad.head(10)
        .merge(
            non_null[["source", "source_url", "puzzle_name"]].drop_duplicates(),
            on=["source", "source_url"],
        )
        .sort_values(["source", "source_url"])
    )
    with pd.option_context("display.max_colwidth", None):
        print(ex[["source", "source_url", "puzzle_name"]].to_string(index=False))

In [ ]:
# === puzzle_name -> source_url Multiplicity ===
name_to_url = (
    non_null.groupby(["source", "puzzle_name"])["source_url"]
    .nunique(dropna=True)
    .reset_index(name="n_urls")
)
bad_rev = name_to_url[name_to_url["n_urls"] > 1]
print(f"puzzle_name values mapping to >1 distinct source_url: {len(bad_rev):,}")
if len(bad_rev) > 0:
    print("\nBreakdown by source:")
    print(bad_rev.groupby("source").size())

    print("\nExample fragmentations (first 10):")
    ex = (
        bad_rev.head(10)
        .merge(
            non_null[["source", "puzzle_name", "source_url"]].drop_duplicates(),
            on=["source", "puzzle_name"],
        )
        .sort_values(["source", "puzzle_name"])
    )
    with pd.option_context("display.max_colwidth", None):
        print(ex[["source", "puzzle_name", "source_url"]].to_string(index=False))

In [ ]:
# === source_url -> puzzle_date Multiplicity ===
# NaN dates are dropped before counting: a URL mapping to one real date plus
# NaN is treated as consistent, not as a conflict. This matters for thebrowser
# where puzzle_date is mostly NaN.
url_to_date = (
    non_null.groupby(["source", "source_url"])["puzzle_date"]
    .nunique(dropna=True)
    .reset_index(name="n_dates")
)
bad_date = url_to_date[url_to_date["n_dates"] > 1]
print(f"source_url values mapping to >1 distinct puzzle_date: {len(bad_date):,}")
if len(bad_date) > 0:
    print("\nBreakdown by source:")
    print(bad_date.groupby("source").size())

    print("\nExample date conflicts (first 10):")
    ex = (
        bad_date.head(10)
        .merge(
            non_null[["source", "source_url", "puzzle_date"]].drop_duplicates(),
            on=["source", "source_url"],
        )
        .sort_values(["source", "source_url"])
    )
    with pd.option_context("display.max_colwidth", None):
        print(ex[["source", "source_url", "puzzle_date"]].to_string(index=False))

---

## §4 — Cross-Source Collisions

Check whether any `source_url` value appears under more than one `source`. Since the ten sources are distinct blogs/publications, a shared URL between two of them is a data-quality red flag — either a scraping artefact or an ambiguity that would make a cross-source grouping key unreliable.

In [ ]:
# === Cross-Source source_url Collisions ===
url_to_sources = (
    non_null.groupby("source_url")["source"]
    .nunique()
    .reset_index(name="n_sources")
)
cross = url_to_sources[url_to_sources["n_sources"] > 1]
print(f"source_url values appearing under >1 source: {len(cross):,}")
if len(cross) > 0:
    print("\nExamples (first 10):")
    ex = (
        cross.head(10)
        .merge(
            non_null[["source_url", "source"]].drop_duplicates(),
            on="source_url",
        )
        .sort_values(["source_url", "source"])
    )
    with pd.option_context("display.max_colwidth", None):
        print(ex[["source_url", "source"]].to_string(index=False))

---

## §5 — Source-Specific Edge Cases

`DATA_RAW.md §4.3` flags several sources whose `source_url` format deserves targeted checks. For each one we verify the assumptions a future `assign_ids.py` would rely on.

- **`thebrowser`** — `source_url` is a file path (`thebrowser/Cryptic NN - MonDDYY Setter.puz`), not a web URL. Check prefix, `.puz` suffix, and overall shape.
- **`natpostcryptic`** — syndicated weekday puzzles coexist with original Saturday puzzles under the same source. Check whether their `source_url` values are distinguishable.
- **`thehinducrosswordcorner`** — two `puzzle_name` formats exist (main series `No NNNNN, ...` and `The Sunday Crossword No NNNNN, ...`). Check whether the two families map to distinct `source_url` patterns.
- **`cru_cryptics`** — `DATA_RAW.md §4` extracts the puzzle number from `source_url` via `r"Cryptic(\d+)\.puz"`. Confirm the pattern matches everywhere.
- **`nytimes`** — `DATA_RAW.md §4` extracts the puzzle date from `source_url` via `r"(\d{8})"`. Confirm the pattern matches everywhere.

In [ ]:
# === thebrowser: File-Path Format ===
bro = df[df["source"] == "thebrowser"]["source_url"]
print(f"thebrowser rows: {len(bro):,}")
print(f"Non-null source_url: {bro.notna().sum():,}")
print(f"Unique source_url: {bro.nunique():,}")

non_null_bro = bro.dropna()
print(
    f"\nStarts with 'thebrowser/': "
    f"{non_null_bro.str.startswith('thebrowser/').sum():,} / {len(non_null_bro):,}"
)
print(
    f"Ends with '.puz': "
    f"{non_null_bro.str.endswith('.puz').sum():,} / {len(non_null_bro):,}"
)

print("\nSample unique source_url values:")
for u in non_null_bro.drop_duplicates().head(10):
    print(f"  {u}")

In [ ]:
# === natpostcryptic: Weekday vs. Saturday by source_url ===
npc = df[df["source"] == "natpostcryptic"]
print(f"natpostcryptic rows: {len(npc):,}")
print(f"Unique source_url values: {npc['source_url'].nunique():,}")

# Partition by puzzle_name weekday prefix (format documented in DATA_RAW.md
# §4.3 starts "Day of week, Month D, Year - Title").
sat_mask = npc["puzzle_name"].str.startswith("Saturday", na=False)
weekday_mask = npc["puzzle_name"].notna() & ~sat_mask
print(f"Saturday puzzle_name rows: {sat_mask.sum():,}")
print(f"Non-Saturday puzzle_name rows: {weekday_mask.sum():,}")

print("\nSample Saturday source_urls:")
for u in npc.loc[sat_mask, "source_url"].dropna().drop_duplicates().head(5):
    print(f"  {u}")

print("\nSample non-Saturday source_urls:")
for u in npc.loc[weekday_mask, "source_url"].dropna().drop_duplicates().head(5):
    print(f"  {u}")

In [ ]:
# === thehinducrosswordcorner: Two puzzle_name Formats vs. source_url ===
hin = df[df["source"] == "thehinducrosswordcorner"]
print(f"thehinducrosswordcorner rows: {len(hin):,}")
print(f"Unique source_url values: {hin['source_url'].nunique():,}")

sunday_mask = hin["puzzle_name"].str.contains(
    "Sunday Crossword", case=False, na=False
)
print(f"'Sunday Crossword' rows: {sunday_mask.sum():,}")
print(f"Main-series rows: {(~sunday_mask & hin['puzzle_name'].notna()).sum():,}")

print("\nSample Sunday Crossword source_urls:")
for u in hin.loc[sunday_mask, "source_url"].dropna().drop_duplicates().head(5):
    print(f"  {u}")

print("\nSample main-series source_urls:")
for u in hin.loc[~sunday_mask, "source_url"].dropna().drop_duplicates().head(5):
    print(f"  {u}")

In [ ]:
# === cru_cryptics: Does source_url Match Cryptic\d+\.puz? ===
cru = df[df["source"] == "cru_cryptics"]["source_url"].dropna()
print(f"cru_cryptics rows with source_url: {len(cru):,}")

matches = cru.str.contains(r"Cryptic\d+\.puz", regex=True)
print(
    f"Rows matching r'Cryptic\\d+\\.puz': "
    f"{matches.sum():,} ({100 * matches.mean():.1f}%)"
)

print("\nSample unique source_url values:")
for u in cru.drop_duplicates().head(10):
    print(f"  {u}")

if (~matches).any():
    print("\nSample NON-matching source_urls:")
    for u in cru[~matches].drop_duplicates().head(10):
        print(f"  {u}")

In [ ]:
# === nytimes: Does source_url Contain an 8-Digit Date? ===
nyt = df[df["source"] == "nytimes"]["source_url"].dropna()
print(f"nytimes rows with source_url: {len(nyt):,}")

matches = nyt.str.contains(r"\d{8}", regex=True)
print(
    f"Rows containing an 8-digit date: "
    f"{matches.sum():,} ({100 * matches.mean():.1f}%)"
)

print("\nSample unique source_url values:")
for u in nyt.drop_duplicates().head(10):
    print(f"  {u}")

if (~matches).any():
    print("\nSample NON-matching source_urls:")
    for u in nyt[~matches].drop_duplicates().head(10):
        print(f"  {u}")

---

## §6 — Recommendation

*To be filled in after running §1–§5 and reading the cell outputs.*

This section should answer, in order:

1. **Can `source_url` be used directly as a grouping key?** Yes / no / with caveats, backed by the §3 one-to-one check results.
2. **What normalisation is needed before use?** List any transformations required based on §2 findings — e.g. lowercasing, trailing-slash stripping, `http` → `https` canonicalisation.
3. **Which sources require special handling?** Enumerate any sources flagged by §1 (high null rate), §4 (cross-source collision), or §5 (source-specific edge cases) that need a fallback grouping strategy or a documented exception.
4. **Which sources are safe to group on `source_url` as-is?** The complement of the above.

---

## §7 — Summary

**What was done:** Loaded `clue_id`, `source`, `puzzle_name`, `source_url`, and `puzzle_date` from `data/clues_raw.csv` (660,613 rows) and ran five diagnostic passes to evaluate `source_url` as a puzzle grouping key — per-source null rates (§1), intra-source formatting drift (§2), one-to-one correspondence with `puzzle_name` and `puzzle_date` (§3), cross-source collisions (§4), and source-specific edge cases documented in `DATA_RAW.md §4.3` (§5).

**Outputs:** None. This notebook writes no files; its purpose is to surface evidence for a design decision.

**Next step:** Fill in the §6 recommendation from the cell outputs above. Those findings feed directly into the design of `assign_ids.py`. This notebook is run-once-and-read — not part of the automated pipeline.